# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [3]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [4]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [7]:
from nltk.stem import RSLPStemmer

stemmer = RSLPStemmer()

# Negations carry the meaning in a bad review ("não recomendo" = do not recommend),
# so they are deliberately kept out of the stopword list.
NEGATIONS = {"não", "nao", "nem", "nunca", "sem"}

_pt = set(stopwords.words("portuguese"))
_pt_ascii = {unidecode.unidecode(w) for w in _pt}
STOPWORDS_PT = (_pt | _pt_ascii) - NEGATIONS


def tokenize_pt(sentence):
    """Lowercase, drop digits/punctuation, tokenize, remove stopwords.
    Accents are kept here on purpose: the RSLP rules are written for accented text."""
    sentence = str(sentence).lower().strip()
    sentence = "".join(ch for ch in sentence if not ch.isdigit())
    sentence = "".join(" " if ch in string.punctuation else ch for ch in sentence)
    tokens = word_tokenize(sentence, language="portuguese")
    return [t for t in tokens if t not in STOPWORDS_PT and len(t) > 2]


def cleaning(sentence):
    """Baseline track: readable Portuguese words, accents folded to ASCII."""
    return " ".join(unidecode.unidecode(t) for t in tokenize_pt(sentence))


def cleaning_stemmed(sentence):
    """Stemmed track: stem first (rules need accents), then fold to ASCII."""
    return " ".join(unidecode.unidecode(stemmer.stem(t)) for t in tokenize_pt(sentence))


# Sanity check on one review
sample = df.loc[0, "full_review"]
print("RAW:", sample[:160])
print("CLEANED:", cleaning(sample)[:160])
print("STEMMED:", cleaning_stemmed(sample)[:160])

RAW:  Recebi bem antes do prazo estipulado.
CLEANED: recebi bem antes prazo estipulado
STEMMED: receb bem ant praz estipul


In [9]:
df["full_review_cleaned"] = df["full_review"].apply(cleaning)
df["full_review_stemmed"] = df["full_review"].apply(cleaning_stemmed)

# How much does stemming actually collapse the vocabulary?
vocab_cleaned = set(" ".join(df["full_review_cleaned"]).split())
vocab_stemmed = set(" ".join(df["full_review_stemmed"]).split())

print(f"vocabulary — cleaned: {len(vocab_cleaned):,}")
print(f"vocabulary — stemmed: {len(vocab_stemmed):,}")
print(f"reduction: {1 - len(vocab_stemmed) / len(vocab_cleaned):.1%}")

df[["review_score", "full_review", "full_review_cleaned", "full_review_stemmed"]].head()

vocabulary — cleaned: 12,827
vocabulary — stemmed: 6,873
reduction: 46.4%


,review_score,full_review,full_review_cleaned,full_review_stemmed
0,5,Recebi bem antes do prazo estipulado.,recebi bem antes prazo estipulado,receb bem ant praz estipul
1,5,Parabéns lojas lannister adorei comprar pela ...,parabens lojas lannister adorei comprar intern...,parabem loj lannist ador compr internet segur ...
2,4,recomendo aparelho eficiente. no site a marca ...,recomendo aparelho eficiente site marca aparel...,recom aparelh efici sit marc aparelh impress d...
3,4,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",pouco travando valor boa,pouc trav val boa
4,5,"Super recomendo Vendedor confiável, produto ok...",super recomendo vendedor confiavel produto ent...,sup recom vend confi produt entreg ant praz


## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [10]:
bad_mask = df["review_score"].isin([1, 2, 3])

n_bad = bad_mask.sum()
share_bad = bad_mask.mean()

print(f"bad reviews (1-3): {n_bad:,} / {len(df):,} → {share_bad:.1%}")
print()
print(df["review_score"].value_counts(normalize=True).sort_index().mul(100).round(1))

bad reviews (1-3): 9,658 / 36,148 → 26.7%

review_score
1    13.4
2     4.6
3     8.7
4    16.1
5    57.2
Name: proportion, dtype: float64


🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [11]:
bad = df[bad_mask].copy().reset_index(drop=True)

print(f"shape: {bad.shape}")
print(f"mean review length: {bad['length_review'].mean():.0f} chars "
      f"(vs {df.loc[~bad_mask, 'length_review'].mean():.0f} for 4-5 star)")
print()
print("top categories among bad reviews:")
print(bad["product_category_name"].value_counts().head(10))

bad[["review_score", "product_category_name", "full_review_cleaned"]].head()

shape: (9658, 8)
mean review length: 96 chars (vs 51 for 4-5 star)

top categories among bad reviews:
product_category_name
cama_mesa_banho           1221
informatica_acessorios     797
moveis_decoracao           743
beleza_saude               716
esporte_lazer              626
relogios_presentes         589
utilidades_domesticas      586
telefonia                  488
automotivo                 370
ferramentas_jardim         318
Name: count, dtype: int64


,review_score,product_category_name,full_review_cleaned
0,1,eletronicos,recebi somente controle midea split estilo fal...
1,3,ferramentas_jardim,comprei duas unidades recebi agora faco
2,3,beleza_saude,produto bom porem veio mim nao condiz foto anu...
3,1,moveis_decoracao,produto inferior mal acabado
4,3,esporte_lazer,entrega prazo


### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [14]:
VECT_PARAMS = dict(
    ngram_range=(1, 2),   # unigrams + bigrams ("nao recomendo")
    max_df=0.75,          # drop terms appearing in >75% of reviews
    max_features=5000,
)

# Track A: readable words
count_vectorizer = CountVectorizer(**VECT_PARAMS)
X_bad = count_vectorizer.fit_transform(bad["full_review_cleaned"])

# Track B: stemmed
count_vectorizer_stem = CountVectorizer(**VECT_PARAMS)
X_bad_stem = count_vectorizer_stem.fit_transform(bad["full_review_stemmed"])

print(f"cleaned  → {X_bad.shape[0]:,} docs × {X_bad.shape[1]:,} features")
print(f"stemmed  → {X_bad_stem.shape[0]:,} docs × {X_bad_stem.shape[1]:,} features")

# How many of the 5000 selected features are actually bigrams?
for name, vec in [("cleaned", count_vectorizer), ("stemmed", count_vectorizer_stem)]:
    feats = vec.get_feature_names_out()
    n_bi = sum(" " in f for f in feats)
    print(f"{name:8} bigrams: {n_bi:,} / {len(feats):,} ({n_bi/len(feats):.0%})")

freq = pd.DataFrame({
    "term": count_vectorizer.get_feature_names_out(),
    "count": X_bad.sum(axis=0).A1,
}).sort_values("count", ascending=False)

print("\ntop 15 unigrams:")
print(freq[~freq["term"].str.contains(" ")].head(15).to_string(index=False))

print("\ntop 15 bigrams:")
print(freq[freq["term"].str.contains(" ")].head(15).to_string(index=False))

cleaned  → 9,658 docs × 5,000 features
stemmed  → 9,658 docs × 5,000 features
cleaned  bigrams: 2,724 / 5,000 (54%)
stemmed  bigrams: 3,367 / 5,000 (67%)

top 15 unigrams:
     term  count
      nao   5816
  produto   5318
   recebi   2352
     veio   2038
  comprei   1702
 entregue   1073
  entrega   1001
   chegou    989
    prazo    742
    porem    693
    ainda    670
      bom    649
   apenas    637
   pedido    617
qualidade    606

top 15 bigrams:
            term  count
      nao recebi    742
  recebi produto    570
     produto nao    539
    produto veio    421
       ainda nao    365
   nao recomendo    317
      nao gostei    253
produto entregue    244
     nota fiscal    244
   recebi apenas    242
  produto chegou    241
    nao entregue    230
        nao veio    230
      nao chegou    179
    comprei dois    166


### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [15]:
LDA_PARAMS = dict(n_components=3, random_state=42, learning_method="batch")

lda = LatentDirichletAllocation(**LDA_PARAMS)
lda.fit(X_bad)

lda_stem = LatentDirichletAllocation(**LDA_PARAMS)
lda_stem.fit(X_bad_stem)

print("cleaned  components_:", lda.components_.shape)
print("stemmed  components_:", lda_stem.components_.shape)
print("perplexity — cleaned:", round(lda.perplexity(X_bad), 1))
print("perplexity — stemmed:", round(lda_stem.perplexity(X_bad_stem), 1))

cleaned  components_: (3, 5000)
stemmed  components_: (3, 5000)
perplexity — cleaned: 1131.8
perplexity — stemmed: 1046.7


#### Belge Karışımı (Konular için)

In [16]:
document_topic_mixture = lda.transform(X_bad)

print("shape:", document_topic_mixture.shape)
pd.DataFrame(
    document_topic_mixture,
    columns=[f"topic_{i}" for i in range(3)],
).head()

shape: (9658, 3)


,topic_0,topic_1,topic_2
0,0.038174,0.033652,0.928174
1,0.028309,0.027827,0.943864
2,0.689404,0.288868,0.021728
3,0.902239,0.049644,0.048117
4,0.083339,0.832869,0.083792


👉 Her inceleme için en önemli konuyu rapor edelim

In [17]:
bad["most_important_topic"] = document_topic_mixture.argmax(axis=1)

print(bad["most_important_topic"].value_counts().sort_index())
print()
print("mean review score per topic:")
print(bad.groupby("most_important_topic")["review_score"].agg(["count", "mean"]).round(2))

# Propagate topic assignment back to the full dataframe (A plan).
# Reviews scored 4-5 get transformed with the same fitted model, so every
# row has a topic; section (4) then filters down to the worst categories anyway.
X_all = count_vectorizer.transform(df["full_review_cleaned"])
df["most_important_topic"] = lda.transform(X_all).argmax(axis=1)

print("\nfull dataframe topic distribution:")
print(df["most_important_topic"].value_counts().sort_index())

most_important_topic
0    3738
1    2701
2    3219
Name: count, dtype: int64

mean review score per topic:
                      count  mean
most_important_topic             
0                      3738  1.73
1                      2701  2.30
2                      3219  1.52

full dataframe topic distribution:
most_important_topic
0     7546
1    24419
2     4183
Name: count, dtype: int64


#### Konu Karışımı (Kelimeler için)

In [18]:
topic_word_mixture = lda.components_

print("shape:", topic_word_mixture.shape)

pd.DataFrame(
    topic_word_mixture,
    index=[f"topic_{i}" for i in range(3)],
    columns=count_vectorizer.get_feature_names_out(),
).iloc[:, :8]

shape: (3, 5000)


,abaixo,aberta,aberto,abri,abri caixa,abri embalagem,abri reclamacao,abrindo
topic_0,6.980569,21.776336,28.174804,14.244197,2.306592,5.429955,0.346289,6.177426
topic_1,3.634234,2.881008,0.350204,0.462643,0.387759,0.360341,0.337993,0.401509
topic_2,0.385198,0.342656,0.474992,34.293160,3.305649,2.209705,15.315718,1.421065


#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [21]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [22]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


### 🔍 Konuların Yorumlanması

LDA, kötü yorumlar (1–3 yıldız, n=9.658) üzerinde üç ayrı şikâyet ekseni buldu:

| Konu | Tema | Belirleyici kelimeler | Yorum sayısı | Ort. puan |
|---|---|---|---|---|
| 0 | **Ürün beklentiyi karşılamadı** | `diferente`, `foto`, `defeito`, `troca`, `produto veio` | 3.738 | 1.73 |
| 1 | **Teslimat & lojistik (ılımlı)** | `entrega`, `prazo`, `frete`, `correios`, `porem` | 2.701 | 2.30 |
| 2 | **Ürün hiç gelmedi / eksik geldi** | `nao recebi`, `apenas`, `ainda`, `dois`, `pedido` | 3.219 | 1.52 |

**Konu 1'de neden olumlu kelimeler var?** `bom`, `bem` ve `antes` kelimelerinin varlığı çelişki değil, kalıbın kendisi: 3 yıldızlı yorumlar tipik olarak *"ürün iyi **ancak** kargo geç geldi"* biçiminde yazılıyor. `porem` (= ancak) kelimesinin ilk sıralarda olması bunu doğruluyor. Ortalama puanın 2.30 ile en yüksek olması da aynı yönde.

**En sert eksen hangisi?** Konu 2 (ort. 1.52). Yani Olist'te müşteriyi en çok kızdıran şey ürünün kötü olması değil, **ürünün hiç ulaşmaması veya eksik ulaşması**. Ham frekanslar da bunu destekliyor: en sık bigram `nao recebi` (742 kez), ardından `recebi apenas` (242) ve `nao entregue` (230).

**Stemli hat ile doğrulama.** Aynı LDA, RSLP ile köklenmiş metinlere ayrıca uygulandı (kelime dağarcığı 12.827 → 6.873, %46,4 azalma). Konu numaraları farklı sırada çıktı ama **üç eksen birebir yeniden üretildi** (cleaned 0 ↔ stemmed 1, cleaned 1 ↔ stemmed 0, cleaned 2 ↔ stemmed 2). Bu, konuların vektörleştirme tercihine bağlı bir tesadüf olmadığını gösteriyor. Stemli hattın tek ek katkısı Konu 2'de yüzeye çıkan `falt` (*falta* = eksik) kökü oldu.

> ⚠️ İki hattın perplexity değerleri (1.131,8 ve 1.046,7) **karşılaştırılabilir değildir**, çünkü farklı kelime uzaylarında hesaplanmıştır. Stemli hattın düşük değeri kalite üstünlüğü olarak okunmamalıdır.

**Metodolojik not.** LDA yalnızca kötü yorumlar üzerinde `fit` edildi; konular bu yorumlardan öğrenildi. Tüm veri kümesine ise `transform` uygulanarak konu ataması yayıldı, böylece (4). bölümdeki kategori bazlı gruplamalar tam veri üzerinde çalışabiliyor.

🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [23]:
print("=" * 22, "CLEANED", "=" * 22)
print_topics(count_vectorizer, lda, 12)

print()
print("=" * 22, "STEMMED", "=" * 22)
print_topics(count_vectorizer_stem, lda_stem, 12)

====================== CLEANED ======================
--------------------
Topic 0:
[('nao', np.float64(2883.99)), ('produto', np.float64(2770.43)), ('veio', np.float64(1631.28)), ('produto veio', np.float64(420.15)), ('comprei', np.float64(404.06)), ('diferente', np.float64(396.19)), ('produto nao', np.float64(360.07)), ('recomendo', np.float64(353.77)), ('defeito', np.float64(327.17)), ('foto', np.float64(308.23)), ('sem', np.float64(304.34)), ('troca', np.float64(302.08))]
--------------------
Topic 1:
[('produto', np.float64(1390.34)), ('nao', np.float64(1128.09)), ('entrega', np.float64(859.19)), ('prazo', np.float64(742.26)), ('bom', np.float64(643.73)), ('chegou', np.float64(532.51)), ('qualidade', np.float64(313.01)), ('bem', np.float64(270.45)), ('antes', np.float64(266.27)), ('frete', np.float64(262.12)), ('porem', np.float64(253.59)), ('correios', np.float64(245.21))]
--------------------
Topic 2:
[('recebi', np.float64(2177.15)), ('nao', np.float64(1804.93)), ('comprei', np

🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [24]:
# Cells 39 and 81 index this as a list of word-lists, not the raw weight matrix.
topic_word_mixture = [
    topic_word(count_vectorizer, lda, topic, 10, with_weights=False)
    for topic in lda.components_
]

TOPIC_LABELS = {
    0: "Ürün beklentiyi karşılamadı (farklı/defolu/foto uyumsuz)",
    1: "Teslimat & lojistik (ılımlı, çoğunlukla 3 yıldız)",
    2: "Ürün hiç gelmedi veya eksik geldi",
}

for i, words in enumerate(topic_word_mixture):
    print(f"topic {i} — {TOPIC_LABELS[i]}")
    print(f"  {words}\n")

topic 0 — Ürün beklentiyi karşılamadı (farklı/defolu/foto uyumsuz)
  ['nao', 'produto', 'veio', 'produto veio', 'comprei', 'diferente', 'produto nao', 'recomendo', 'defeito', 'foto']

topic 1 — Teslimat & lojistik (ılımlı, çoğunlukla 3 yıldız)
  ['produto', 'nao', 'entrega', 'prazo', 'bom', 'chegou', 'qualidade', 'bem', 'antes', 'frete']

topic 2 — Ürün hiç gelmedi veya eksik geldi
  ['recebi', 'nao', 'comprei', 'produto', 'entregue', 'nao recebi', 'apenas', 'pedido', 'dois', 'ainda']



In [25]:
df["most_important_words"] = df["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [26]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
0,e64fb393e7b32834bb789ff8bb30750e,5,ferramentas_jardim,recebi bem antes prazo estipulado,1,"[produto, nao, entrega, prazo, bom, chegou, qu..."
1,f7c4243c7fe1938f181bec41a392bdeb,5,esporte_lazer,parabens lojas lannister adorei comprar intern...,1,"[produto, nao, entrega, prazo, bom, chegou, qu..."
2,8670d52e15e00043ae7de4c01cc2fe06,4,eletroportateis,recomendo aparelho eficiente site marca aparel...,0,"[nao, produto, veio, produto veio, comprei, di..."
3,4b49719c8a200003f700d3d986ea1a19,4,beleza_saude,pouco travando valor boa,1,"[produto, nao, entrega, prazo, bom, chegou, qu..."
4,3948b09f7c818e2d86c9a546758b2335,5,informatica_acessorios,super recomendo vendedor confiavel produto ent...,1,"[produto, nao, entrega, prazo, bom, chegou, qu..."


## (3.4) Pipeline Tf-Idf ve LDA

In [27]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [28]:
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    TfidfVectorizer(**VECT_PARAMS),
    LatentDirichletAllocation(**LDA_PARAMS),
)

pipeline.fit(bad["full_review_cleaned"])
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('latentdirichletallocation', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.75
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",5000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only 

💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [29]:
pipeline._final_estimator

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",3
,"random_state random_state: int, RandomState instance or None, default=NonePass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0


In [30]:
pipeline._final_estimator.components_

array([[3.67949131, 5.74065618, 6.7331452 , ..., 0.49295254, 4.59626046,
        6.92073882],
       [0.34937068, 1.78695054, 1.23854937, ..., 2.62060987, 0.34556227,
        0.36585731],
       [0.35041674, 0.34364213, 0.88746557, ..., 0.40184082, 4.19050485,
        0.35824555]], shape=(3, 5000))

Pipeline ile **Belge Karışımı**:

In [31]:
document_topic_mixture_pipe = pipeline.transform(bad["full_review_cleaned"])

print("shape:", document_topic_mixture_pipe.shape)
pd.DataFrame(
    document_topic_mixture_pipe,
    columns=[f"topic_{i}" for i in range(3)],
).head()

shape: (9658, 3)


,topic_0,topic_1,topic_2
0,0.092292,0.094069,0.813639
1,0.082191,0.080302,0.837507
2,0.398686,0.523490,0.077824
3,0.379465,0.519237,0.101298
4,0.126663,0.746020,0.127317


Pipeline ile **Konu Karışımı**:

In [34]:
pipe_vectorizer = pipeline.named_steps["tfidfvectorizer"]
pipe_lda = pipeline._final_estimator

topic_word_mixture_pipe = pipe_lda.components_
print("shape:", topic_word_mixture_pipe.shape)

print_topics(pipe_vectorizer, pipe_lda, 12)

# Does TF-IDF weighting reproduce the same three axes as raw counts?
for i in range(3):
    count_words = set(topic_word(count_vectorizer, lda, lda.components_[i], 10, with_weights=False))
    for j in range(3):
        pipe_words = set(topic_word(pipe_vectorizer, pipe_lda, pipe_lda.components_[j], 10, with_weights=False))
        overlap = count_words & pipe_words
        if len(overlap) >= 4:
            print(f"count topic {i}  ↔  tfidf topic {j}   ({len(overlap)}/10 shared: {sorted(overlap)})")

shape: (3, 5000)
--------------------
Topic 0:
[('produto', np.float64(186.62)), ('nao', np.float64(183.25)), ('veio', np.float64(151.65)), ('gostei', np.float64(90.32)), ('recomendo', np.float64(78.39)), ('diferente', np.float64(76.71)), ('boa', np.float64(74.21)), ('produto veio', np.float64(65.39)), ('foto', np.float64(60.4)), ('nao gostei', np.float64(57.85)), ('quero', np.float64(49.49)), ('sem', np.float64(46.97))]
--------------------
Topic 1:
[('produto', np.float64(151.7)), ('entrega', np.float64(142.4)), ('nao', np.float64(134.22)), ('prazo', np.float64(130.23)), ('qualidade', np.float64(87.88)), ('bom', np.float64(75.39)), ('chegou', np.float64(75.0)), ('bem', np.float64(58.67)), ('frete', np.float64(54.27)), ('antes', np.float64(53.49)), ('demorou', np.float64(52.36)), ('achei', np.float64(50.06))]
--------------------
Topic 2:
[('recebi', np.float64(282.09)), ('nao', np.float64(185.1)), ('comprei', np.float64(171.94)), ('nao recebi', np.float64(163.2)), ('produto', np.floa

## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [ ]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [ ]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [ ]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [ ]:
worst_products_reviews["most_important_topic"].value_counts()

In [ ]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

In [ ]:
[topic_word_mixture[i] for i in bad_frequency]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [ ]:
from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [ ]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

### (5.2) En kötü satıcılar tarafından satılan ürünler

In [ ]:
products = Product().get_training_data() [["product_id", "category"]]
products

❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [ ]:
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

In [ ]:
sellers_product_category.groupby("seller_id").count()

### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [ ]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [ ]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

In [ ]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [ ]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.